# Redes de pases en fútbol

Este notebook integra la intuición del caso de redes de pases: convertir una secuencia de pases en una red, donde los jugadores son nodos y los pases son enlaces.

La versión original del caso se apoyaba en el artículo de Dato-Futbol ["Redes de pases en el fútbol"](https://medium.com/datos-y-ciencia/redes-de-pases-en-el-fútbol-f014e6fd9f76) y en el repositorio [`Dato-Futbol/passing-networks`](https://github.com/Dato-Futbol/passing-networks). Aquí usamos una versión pedagógica autocontenida para que puedas ejecutar el flujo sin depender de descargas externas.

Al final se muestra cómo esta lógica se conecta con datos reales de StatsBomb o Metrica Sports.

## Qué vamos a hacer

1. Cargar una tabla de pases simulados.
2. Filtrar pases completados.
3. Construir una `edge list`: quién le pasa a quién y cuántas veces.
4. Construir una tabla de nodos: posición media y participación de cada jugador.
5. Crear una matriz de adyacencia.
6. Dibujar la red sobre una cancha.
7. Interpretar qué jugadores y conexiones estructuran el juego.

In [ ]:
# Import the minimum set of libraries for a reproducible teaching notebook.
from pathlib import Path
from html import escape

import pandas as pd


## 1. Cargar los pases

Cada fila representa un pase observado durante un partido. Para simplificar, usamos un solo equipo y coordenadas en una cancha de `105 x 68` metros.

En datos reales, esta tabla podría venir de una fuente de eventos como StatsBomb o de datos de tracking como Metrica Sports.

In [ ]:
# Load the demo pass event data.
DATA_PATH = Path("pases_futbol_demo.csv")
passes = pd.read_csv(DATA_PATH)

passes.head(8)

## 2. Quedarnos con pases completados

Para una red básica de pases suele interesar mirar los pases que efectivamente conectaron a dos jugadores. Los pases incompletos también pueden analizarse, pero responden otra pregunta: riesgo, error, presión o pérdida.

In [ ]:
# Keep completed passes for the network construction.
completed = passes.query("outcome == 'Complete'").copy()

summary = pd.DataFrame({
    "metric": ["total_passes", "completed_passes", "completion_rate"],
    "value": [
        len(passes),
        len(completed),
        round(len(completed) / len(passes), 3),
    ],
})

summary

## 3. Construir la lista de aristas

La `edge list` es una de las formas más simples de representar una red. En este caso, cada fila dice:

- jugador de origen;
- jugador receptor;
- cantidad de pases entre ambos;
- posición promedio desde donde sale y hacia donde llega esa conexión.

In [ ]:
# Build a directed weighted edge list.
edges = (
    completed
    .groupby(["from_player", "to_player"], as_index=False)
    .agg(
        pass_count=("minute", "size"),
        x_start=("x_start", "mean"),
        y_start=("y_start", "mean"),
        x_end=("x_end", "mean"),
        y_end=("y_end", "mean"),
    )
    .sort_values("pass_count", ascending=False)
)

edges.head(10)

## 4. Construir la tabla de nodos

Los nodos son los jugadores. Para ubicarlos en la cancha, usamos una posición promedio combinando dónde iniciaron pases y dónde recibieron pases.

El tamaño del nodo se relaciona con su participación: pases dados más pases recibidos.

In [ ]:
# Estimate player positions from pass origins and receptions.
origins = completed[["from_player", "x_start", "y_start"]].rename(
    columns={"from_player": "player", "x_start": "x", "y_start": "y"}
)
receptions = completed[["to_player", "x_end", "y_end"]].rename(
    columns={"to_player": "player", "x_end": "x", "y_end": "y"}
)
positions = pd.concat([origins, receptions], ignore_index=True)

node_positions = (
    positions
    .groupby("player", as_index=False)
    .agg(x=("x", "mean"), y=("y", "mean"))
)

out_stats = completed.groupby("from_player").size().rename("passes_given")
in_stats = completed.groupby("to_player").size().rename("passes_received")

nodes = (
    node_positions
    .merge(out_stats, left_on="player", right_index=True, how="left")
    .merge(in_stats, left_on="player", right_index=True, how="left")
    .fillna({"passes_given": 0, "passes_received": 0})
)

nodes["participation"] = nodes["passes_given"] + nodes["passes_received"]
nodes["participation_share"] = nodes["participation"] / nodes["participation"].sum()

nodes.sort_values("participation", ascending=False)

## 5. Mirar la matriz de adyacencia

La misma red puede verse como matriz. Las filas indican quién entrega el pase y las columnas quién lo recibe.

Esta vista es menos visual que el grafo, pero ayuda a reconocer pares fuertes y a preparar análisis más formales.

In [ ]:
# Create an adjacency matrix from the edge list.
adjacency = (
    edges
    .pivot(index="from_player", columns="to_player", values="pass_count")
    .fillna(0)
    .astype(int)
)

adjacency

## 6. Dibujar la red sobre una cancha

El gráfico combina tres ideas:

- la posición del jugador en la cancha;
- el tamaño del nodo según participación;
- el grosor de la flecha según cantidad de pases.

Para no saturar la imagen, mostramos solo conexiones con al menos `2` pases.

In [ ]:
# Build a simple SVG pitch and passing network without external plotting libraries.
def scale_x(x, margin=35, pitch_length=105, width=900):
    return margin + (x / pitch_length) * (width - 2 * margin)


def scale_y(y, margin=35, pitch_width=68, height=580):
    return margin + ((pitch_width - y) / pitch_width) * (height - 2 * margin)


def make_passing_network_svg(nodes_df, edges_df, min_passes=2, out_path="red_pases_demo.svg"):
    width, height = 900, 580
    margin = 35
    pitch_w = width - 2 * margin
    pitch_h = height - 2 * margin
    svg = []
    svg.append(f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">')
    svg.append('<rect width="100%" height="100%" rx="24" fill="#f6faf8"/>')
    svg.append(f'<rect x="{margin}" y="{margin}" width="{pitch_w}" height="{pitch_h}" fill="#edf7f0" stroke="#8aa99a" stroke-width="2"/>')
    svg.append(f'<line x1="{width/2}" y1="{margin}" x2="{width/2}" y2="{height-margin}" stroke="#c4d4cc" stroke-width="2"/>')
    svg.append(f'<circle cx="{width/2}" cy="{height/2}" r="62" fill="none" stroke="#c4d4cc" stroke-width="2"/>')
    svg.append(f'<rect x="{margin}" y="{scale_y(54.16)}" width="{(16.5/105)*pitch_w}" height="{(40.32/68)*pitch_h}" fill="none" stroke="#c4d4cc" stroke-width="2"/>')
    svg.append(f'<rect x="{scale_x(88.5)}" y="{scale_y(54.16)}" width="{(16.5/105)*pitch_w}" height="{(40.32/68)*pitch_h}" fill="none" stroke="#c4d4cc" stroke-width="2"/>')
    svg.append('<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto" markerUnits="strokeWidth"><path d="M0,0 L0,6 L9,3 z" fill="#2563eb" /></marker></defs>')

    plot_edges = edges_df.query("pass_count >= @min_passes").copy()
    max_count = max(plot_edges["pass_count"].max(), 1)
    for _, row in plot_edges.iterrows():
        x1, y1 = scale_x(row["x_start"]), scale_y(row["y_start"])
        x2, y2 = scale_x(row["x_end"]), scale_y(row["y_end"])
        stroke_width = 1.4 + 5.0 * (row["pass_count"] / max_count)
        svg.append(
            f'<line x1="{x1:.1f}" y1="{y1:.1f}" x2="{x2:.1f}" y2="{y2:.1f}" '
            f'stroke="#2563eb" stroke-opacity="0.42" stroke-width="{stroke_width:.1f}" marker-end="url(#arrow)"/>'
        )

    max_participation = max(nodes_df["participation"].max(), 1)
    for _, row in nodes_df.iterrows():
        cx, cy = scale_x(row["x"]), scale_y(row["y"])
        radius = 9 + 18 * (row["participation"] / max_participation)
        label = escape(str(row["player"]))
        svg.append(f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="{radius:.1f}" fill="#0f766e" fill-opacity="0.92" stroke="white" stroke-width="2"/>')
        svg.append(f'<text x="{cx:.1f}" y="{cy - radius - 8:.1f}" text-anchor="middle" font-family="Arial, sans-serif" font-size="13" font-weight="700" fill="#102133">{label}</text>')

    svg.append('<text x="35" y="558" font-family="Arial, sans-serif" font-size="13" fill="#5f7288">Nodo = jugador; tamaño = participación; flecha = pases completados; grosor = frecuencia.</text>')
    svg.append('</svg>')
    Path(out_path).write_text("\n".join(svg), encoding="utf-8")
    return out_path

svg_path = make_passing_network_svg(nodes, edges, min_passes=2)
print(f"Saved: {svg_path}")


![Red de pases demo](red_pases_demo.svg)


## 7. Interpretar la red

Una red de pases no dice automáticamente quién jugó “mejor”. Lo que muestra es una estructura de circulación:

- quién participa más en la posesión;
- qué conexiones se repiten;
- si el equipo sale por una banda, por el centro o alterna rutas;
- si hay jugadores puente que conectan defensa, mediocampo y ataque.

In [ ]:
# Create a compact interpretation table.
player_summary = nodes.copy()
player_summary["passes_given"] = player_summary["passes_given"].astype(int)
player_summary["passes_received"] = player_summary["passes_received"].astype(int)
player_summary["participation"] = player_summary["participation"].astype(int)
player_summary["participation_share"] = player_summary["participation_share"].round(3)

player_summary = player_summary[[
    "player", "passes_given", "passes_received", "participation", "participation_share"
]].sort_values("participation", ascending=False)

player_summary

In [ ]:
# Identify the strongest pass combinations.
strongest_connections = edges[["from_player", "to_player", "pass_count"]].head(8)
strongest_connections

## 8. Cómo se conecta con el repositorio original

El repositorio clonado (`Dato-Futbol/passing-networks`) hace una versión más completa de esta misma idea, especialmente en R:

- usa datos reales de eventos de StatsBomb o tracking de Metrica Sports;
- ubica jugadores en la cancha usando información de origen/destino de pases;
- permite configurar dirección de pases, tamaño de nodos, grosor de aristas, convex hull y estilo del campo;
- genera visualizaciones listas para análisis futbolístico.

Este notebook reduce el problema a su esqueleto para que sea más transparente:

1. tabla de pases;
2. lista de aristas;
3. tabla de nodos;
4. matriz de adyacencia;
5. visualización;
6. interpretación.

Si luego queremos usar datos reales, el siguiente paso es reemplazar `pases_futbol_demo.csv` por una tabla de eventos con columnas equivalentes.

In [ ]:
# Save derived tables for a future interactive site or handout.
edges.to_csv("red_pases_edge_list.csv", index=False)
nodes.to_csv("red_pases_nodes.csv", index=False)
adjacency.to_csv("red_pases_adjacency_matrix.csv")

print("Saved: red_pases_edge_list.csv")
print("Saved: red_pases_nodes.csv")
print("Saved: red_pases_adjacency_matrix.csv")

## Preguntas para cerrar

- ¿Qué jugador parece organizar la circulación del equipo? ¿Por qué?
- ¿La red sugiere una salida más central o por bandas?
- ¿Qué se perdería si solo miráramos cantidad total de pases y no la estructura de conexiones?
- ¿Qué cambiaría si incluyéramos pases incompletos?
- ¿Qué variable adicional sería útil para enriquecer esta red: tiempo, zona, presión rival, valor esperado del pase u otra?